# RetailPulse – Exploratory Data Analysis (EDA)

**Project:** RetailPulse – AI-Powered Customer Analytics & Demand Forecasting  
**Internship:** Zidio Development Data Science Internship (June–September 2026)  
**Dataset:** Online Retail II (UCI ML Repository)  

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Setup & Data Loading |
| 2 | Data Quality Check |
| 3 | Revenue & Sales Trends |
| 4 | Customer Analysis |
| 5 | Product Analysis |
| 6 | RFM Feature Engineering |
| 7 | Customer Segmentation (K-Means) |
| 8 | Demand Forecasting Preview |
| 9 | Churn Feature Exploration |
| 10 | Key Insights Summary |

## 1. Setup & Data Loading

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Make the project root importable from within notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Notebook display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('✅ Libraries loaded')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

In [ ]:
from src.data_preprocessing import generate_synthetic_data, clean_data, build_rfm, build_daily_sales

# Generate synthetic data (swap with load_raw_data() once you have the real dataset)
raw = generate_synthetic_data(n_customers=1000, n_transactions=50_000)
print(f'Raw data shape: {raw.shape}')
raw.head()

## 2. Data Quality Check

In [ ]:
print('=== Raw Data Info ===')
print(f'Shape: {raw.shape}')
print(f'\nData types:')
print(raw.dtypes)
print(f'\nMissing values:')
print(raw.isnull().sum())

In [ ]:
# Count cancellations (InvoiceNo starts with C)
n_cancel = raw['InvoiceNo'].astype(str).str.startswith('C').sum()
n_null_cust = raw['CustomerID'].isnull().sum()
n_neg_qty = (raw['Quantity'] <= 0).sum()
n_zero_price = (raw['UnitPrice'] <= 0).sum()

print('=== Data Quality Issues ===')
print(f'Cancellations (InvoiceNo starts with C) : {n_cancel:,}  ({n_cancel/len(raw)*100:.1f}%)')
print(f'Missing CustomerID                       : {n_null_cust:,}  ({n_null_cust/len(raw)*100:.1f}%)')
print(f'Non-positive Quantity                    : {n_neg_qty:,}')
print(f'Zero or negative UnitPrice               : {n_zero_price:,}')

In [ ]:
# Clean the data
df = clean_data(raw)
print(f'\nCleaned data shape: {df.shape}')
df.head()

## 3. Revenue & Sales Trends

In [ ]:
# Basic revenue KPIs
total_revenue   = df['TotalAmount'].sum()
total_orders    = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
avg_order_value = df['TotalAmount'].mean()

print('=== Revenue KPIs ===')
print(f'Total Revenue       : £{total_revenue:,.2f}')
print(f'Total Orders        : {total_orders:,}')
print(f'Unique Customers    : {total_customers:,}')
print(f'Avg Order Value     : £{avg_order_value:.2f}')

In [ ]:
# Monthly revenue trend
monthly = (df.set_index('InvoiceDate')
             .resample('ME')['TotalAmount']
             .sum()
             .reset_index())

fig = px.line(monthly, x='InvoiceDate', y='TotalAmount', markers=True,
              title='Monthly Revenue Trend',
              labels={'TotalAmount': 'Revenue (£)', 'InvoiceDate': 'Month'})
fig.update_traces(line_color='#0969da', line_width=2)
fig.show()

In [ ]:
# Revenue by day of week
dow_map = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
df['DayName'] = df['InvoiceDate'].dt.dayofweek.map(dow_map)
dow = df.groupby('DayName')['TotalAmount'].sum().reindex(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(dow.index, dow.values, color='#0969da', edgecolor='white')
ax.set_title('Revenue by Day of Week', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (£)')
plt.tight_layout()
plt.show()

## 4. Customer Analysis

In [ ]:
# Revenue by country
country_rev = (df.groupby('Country')['TotalAmount']
                 .sum()
                 .sort_values(ascending=False)
                 .head(10)
                 .reset_index())

fig = px.bar(country_rev, x='TotalAmount', y='Country', orientation='h',
             title='Top 10 Countries by Revenue',
             color='TotalAmount', color_continuous_scale='Blues',
             labels={'TotalAmount': 'Revenue (£)'})
fig.update_layout(coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Customer purchase frequency distribution
purchase_freq = df.groupby('CustomerID')['InvoiceNo'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(purchase_freq, bins=40, color='#0969da', edgecolor='white')
axes[0].set_title('Purchase Frequency Distribution')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Customers')

# Revenue per customer distribution (log scale for skewed data)
cust_rev = df.groupby('CustomerID')['TotalAmount'].sum()
axes[1].hist(cust_rev, bins=40, color='#1a7f37', edgecolor='white')
axes[1].set_title('Revenue per Customer Distribution')
axes[1].set_xlabel('Total Revenue (£)')
axes[1].set_ylabel('Customers')

plt.tight_layout()
plt.show()

print(f'Median purchases per customer : {purchase_freq.median():.1f}')
print(f'Mean revenue per customer     : £{cust_rev.mean():,.2f}')

## 5. Product Analysis

In [ ]:
# Top 10 products by revenue
top_products = (df.groupby('Description')['TotalAmount']
                  .sum()
                  .nlargest(10)
                  .reset_index())

fig = px.bar(top_products.sort_values('TotalAmount'), x='TotalAmount', y='Description',
             orientation='h', title='Top 10 Products by Revenue',
             color='TotalAmount', color_continuous_scale='Blues',
             labels={'TotalAmount': 'Revenue (£)'})
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Order value distribution (remove top 1% outliers for readability)
q99 = df['TotalAmount'].quantile(0.99)
order_vals = df[df['TotalAmount'] < q99]['TotalAmount']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(order_vals, bins=60, color='#9a3412', edgecolor='white', alpha=0.85)
ax.axvline(order_vals.mean(), color='red', linestyle='--', label=f'Mean: £{order_vals.mean():.2f}')
ax.set_title('Order Value Distribution (excluding top 1% outliers)')
ax.set_xlabel('Order Value (£)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 6. RFM Feature Engineering

In [ ]:
rfm = build_rfm(df)
print(f'RFM shape: {rfm.shape}')
rfm.head()

In [ ]:
# RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes,
                          ['Recency', 'Frequency', 'Monetary'],
                          ['#0969da', '#1a7f37', '#9a3412']):
    ax.hist(rfm[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Customers')

plt.suptitle('RFM Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Segment counts
seg_counts = rfm['Segment'].value_counts().reset_index()
seg_counts.columns = ['Segment', 'Count']

fig = px.pie(seg_counts, names='Segment', values='Count',
             title='Customer Segment Distribution',
             color_discrete_sequence=px.colors.qualitative.Pastel,
             hole=0.4)
fig.update_traces(textposition='outside', textinfo='label+percent')
fig.show()

## 7. Customer Segmentation (K-Means)

In [ ]:
from src.segmentation import preprocess_rfm, find_optimal_k, run_kmeans, interpret_clusters

X_scaled, scaler, features = preprocess_rfm(rfm)
best_k = find_optimal_k(X_scaled, k_range=range(2, 9))
print(f'Best K: {best_k}')

In [ ]:
rfm, sil_score = run_kmeans(X_scaled, rfm, k=best_k)
rfm, cluster_summary = interpret_clusters(rfm)
print(f'Silhouette Score: {sil_score:.4f}')
cluster_summary

In [ ]:
# 3D scatter of RFM clusters
sample = rfm.sample(min(2000, len(rfm)), random_state=42)
fig = px.scatter_3d(sample, x='Recency', y='Frequency', z='Monetary',
                    color='Business_Segment', opacity=0.65,
                    color_discrete_sequence=px.colors.qualitative.Bold,
                    title='Customer Segments in RFM Space (3D)')
fig.update_layout(height=550)
fig.show()

## 8. Demand Forecasting Preview

In [ ]:
daily = build_daily_sales(df)
print(f'Daily sales shape: {daily.shape}')
print(f'Date range: {daily["ds"].min().date()} → {daily["ds"].max().date()}')
daily.tail()

In [ ]:
# Plot daily revenue with rolling averages
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily['ds'], y=daily['y'],
                          name='Daily Revenue', line=dict(color='lightblue', width=1)))
fig.add_trace(go.Scatter(x=daily['ds'], y=daily['rolling_7d_mean'],
                          name='7-Day Rolling Avg', line=dict(color='#0969da', width=2)))
fig.add_trace(go.Scatter(x=daily['ds'], y=daily['rolling_30d_mean'],
                          name='30-Day Rolling Avg', line=dict(color='#cf222e', width=2)))
fig.update_layout(title='Daily Revenue with Rolling Averages',
                  xaxis_title='Date', yaxis_title='Revenue (£)',
                  hovermode='x unified', height=420)
fig.show()

## 9. Churn Feature Exploration

In [ ]:
# Simple churn proxy from RFM (no file I/O needed)
# Customers with R_Score ≤ 2 are considered "at risk of churn"
rfm['churned_proxy'] = (rfm['R_Score'] <= 2).astype(int)
churn_rate = rfm['churned_proxy'].mean() * 100
print(f'Proxy churn rate: {churn_rate:.1f}%')
print(rfm['churned_proxy'].value_counts())

In [ ]:
# Correlation heatmap of RFM features
rfm_num = rfm[['Recency', 'Frequency', 'Monetary',
                'R_Score', 'F_Score', 'M_Score', 'RFM_Score']]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rfm_num.corr(), annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax)
ax.set_title('RFM Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Key Insights Summary

In [ ]:
print('=' * 55)
print('  RETAILPULSE – KEY EDA INSIGHTS')
print('=' * 55)
print(f'  Total Revenue          : £{df["TotalAmount"].sum():>12,.2f}')
print(f'  Total Customers        : {df["CustomerID"].nunique():>12,}')
print(f'  Total Transactions     : {df["InvoiceNo"].nunique():>12,}')
print(f'  Avg Order Value        : £{df["TotalAmount"].mean():>12.2f}')
print(f'  Top Country            : {df.groupby("Country")["TotalAmount"].sum().idxmax():>20}')
print(f'  Proxy Churn Rate       : {churn_rate:>11.1f}%')
print(f'  K-Means Best K         : {best_k:>12}')
print(f'  Silhouette Score       : {sil_score:>12.4f}')
print('=' * 55)
print()
print('Next steps:')
print('  1. Run run_pipeline.py to train all models')
print('  2. Launch: streamlit run app.py')